<a href="https://colab.research.google.com/github/smduarte/spbd-2627/blob/main/docs/labs/lab5/SPBD_Labs_spark2_exercises_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 - Python Spark Dataframes Exercises Solutions


In [61]:
#@title Preparation
!wget -q -O os_maias.txt https://www.dropbox.com/s/n24v0z7y79np319/os_maias.txt?dl=0

##1. Sorted Word Frequency

1.1) Create a [Spark Dataframes](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html) program that counts the number of occurrences of each word in "Os Maias" novel;

a) No sorting required, words and frequency can appear in any order;

b) Sorted by word, in reverse alphabetical order;

c) Sorted by frequency (the words with higher occurrence first).

Note that the sorting should be performed as a transformation (i.e. it should produce an RDD)...

In [62]:
#@title Exercise 1.1a - WordFrequency: No Sorting
import re
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('wordfrequency1.1a)').getOrCreate()

try :
  df = spark.read.text('os_maias.txt') \
        .withColumn('line', lower('value')) \
        .where( length('line') > 0) \
        .withColumn('words', regexp_extract_all('line', lit(r"(?U)[^\W_]+"),0)) \
        .withColumn('word', explode('words')) \
        .groupBy('word').count()

  df.show(5)

except Exception as err:
  print(err)

+----------+-----+
|      word|count|
+----------+-----+
|    linhas|   16|
|     vinte|   16|
|     podia|  113|
|anunciando|   11|
|  pastores|    2|
+----------+-----+
only showing top 5 rows


In [63]:
#@title Exercise 1.1b - WordFrequency: Sorted by word, alphabetically

import re
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('wordfrequency1.1b)').getOrCreate()

try :
  df = spark.read.text('os_maias.txt') \
        .withColumn('line', lower('value')) \
        .where( length('line') > 0) \
        .withColumn('words', regexp_extract_all('line', lit(r"(?U)[^\W_]+"),0)) \
        .withColumn('word', explode('words')) \
        .groupBy('word').count() \
        .orderBy('word')

  df.show(5)

except Exception as err:
  print(err)

+----+-----+
|word|count|
+----+-----+
|   0|    2|
|   1|    1|
|  15|    1|
|1815|    1|
|1830|    3|
+----+-----+
only showing top 5 rows


In [64]:
#@title Exercise 1.1c - WordFrequency: Sorted by Frequency

import re
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('wordfrequency1.1c)').getOrCreate()

try :
  df = spark.read.text('os_maias.txt') \
        .withColumn('line', lower('value')) \
        .where( length('line') > 0) \
        .withColumn('words', regexp_extract_all('line', lit(r"(?U)[^\W_]+"),0)) \
        .withColumn('word', explode('words')) \
        .groupBy('word').count() \
        .orderBy('count', ascending=False)

  df.show(5)

except Exception as err:
  print(err)

+----+-----+
|word|count|
+----+-----+
|  de| 8505|
|   o| 7867|
|   a| 7417|
|   e| 6231|
| que| 5280|
+----+-----+
only showing top 5 rows


1.2) Create a Spark Dataframes program that computes the top 10 most used words in "Os Maias" novel.

In [68]:
#@title Exercise 1.2 - Top 10 Words by frequency (version 1)

import re
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('top 10)').getOrCreate()

try :
  df = spark.read.text('os_maias.txt') \
        .withColumn('line', lower('value')) \
        .where( length('line') > 0) \
        .withColumn('words', regexp_extract_all('line', lit(r"(?U)[^\W_]+"),0)) \
        .withColumn('word', explode('words')) \
        .groupBy('word').count() \
        .orderBy('count', ascending=False) \
        .limit(10)

  df.show(10)
except Exception as err:
  print(err)

+----+-----+
|word|count|
+----+-----+
|  de| 8505|
|   o| 7867|
|   a| 7417|
|   e| 6231|
| que| 5280|
|  um| 3191|
|  se| 3132|
| com| 2872|
|  do| 2588|
| uma| 2298|
+----+-----+



In [69]:
#@title Exercise 1.2 - Top 10 Words by frequency (version 2)

import re
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('top 10)').getOrCreate()

def partial_topk_arrow(iterator):
    import pyarrow as pa

    freq = {}
    for batch in iterator:
        for w in batch.column('word').to_pylist():
          freq[w] = freq.get(w, 0) + 1

    values = sorted( freq.items(), key = lambda x : -x[1])[0:10]
    words = [w for w, c in values]
    counts = [c for w, c in values]

    yield pa.record_batch([
        pa.array(words),
        pa.array(counts)
    ], names=['word', 'count'])

try :
  df = spark.read.text('os_maias.txt') \
        .withColumn('line', lower('value')) \
        .where( length('line') > 0) \
        .withColumn('words', regexp_extract_all('line', lit(r"(?U)[^\W_]+"),0)) \
        .withColumn('word', explode('words')) \
        .repartition('word') \
        .mapInArrow(partial_topk_arrow, schema='word string, count long') \
        .orderBy('count', ascending = False) \
        .limit(10)

  df.show(10)
except Exception as err:
  print(err)

+----+-----+
|word|count|
+----+-----+
|  de| 8505|
|   o| 7867|
|   a| 7417|
|   e| 6231|
| que| 5280|
|  um| 3191|
|  se| 3132|
| com| 2872|
|  do| 2588|
| uma| 2298|
+----+-----+



##2. Weblog Analysis

Consider a set of log files captured during a DDOS (*Distributed Denial of Service*) attack, containing information for the web accesses performed during the attack to the server.

The log files contain text lines as shown below, with TAB as the separator:

date |IP_source | status_code | operation | URL | execution time |
-|-|-|-|-|-
timestamp  | string | int | string | string| float |
2016-12-06T08:58:35.318+0000|37.139.9.11|404|GET|/codemove/TTCENCUFMH3C|0.026

In [70]:
#@title Download the dataset
!wget -q -O weblog.txt https://www.dropbox.com/s/0r8902uj9yum7dg/web.log?dl=0
!head -3 weblog.txt

2016-12-06T08:58:35.318+0000 37.139.9.11 404 GET /codemove/TTCENCUFMH3C 0.026  
2016-12-06T08:58:35.356+0000 178.22.148.122 404 GET /codemove/PSO83TYKET12 0.088  
2016-12-06T08:58:35.357+0000 178.22.148.122 404 GET /codemove/PSO83TYKET12 0.088  


2.1. Count the number of unique IP addresses involved in the DDOS attack.


In [71]:
#@title Exercise 2.1 - Unique IPs

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('weblog2.1)').getOrCreate()

try :
  df_lines = spark.read.text('weblog.txt') \
                .withColumn('parts', split('value', ' '))

  # Extract array elements and cast to desired types
  df = df_lines.select(
    col('parts')[0].cast('timestamp').alias('timestamp'),
    col('parts')[1].alias('ip'),
    col('parts')[2].cast('integer').alias('status'),
    col('parts')[3].alias('method'),
    col('parts')[4].alias('url'),
    col('parts')[5].cast('double').alias('exec_time')
  )

  df.printSchema()

  unique_ips = df.select('ip') \
        .distinct()

  print(f"Unique Ips: {unique_ips.count()}")

except Exception as err:
  print(err)

root
 |-- timestamp: timestamp (nullable = true)
 |-- ip: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- method: string (nullable = true)
 |-- url: string (nullable = true)
 |-- execution_time: double (nullable = true)

Unique Ips: 167


2.2. For each interval of 10 seconds, provide the following information: [number of requests, average execution time, maximum time, minimum time]

In [73]:
#@title Exercise 2.2 - 10-second interval stats

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('weblog2.2)').getOrCreate()

try :
  df_lines = spark.read.text('weblog.txt') \
                .withColumn('parts', split('value', ' '))

  # Extract array elements and cast to desired types
  df = df_lines.select(
    col('parts')[0].cast('timestamp').alias('timestamp'),
    col('parts')[1].alias('ip'),
    col('parts')[2].cast('integer').alias('status'),
    col('parts')[3].alias('method'),
    col('parts')[4].alias('url'),
    col('parts')[5].cast('double').alias('exec_time')
  )

  df = df.groupBy(window('timestamp', '10 seconds').start) \
          .agg(
              count('*').alias('requests'),
              mean('exec_time').alias('avg'),
              max('exec_time').alias('max'),
              min('exec_time').alias('min')) \
          .orderBy('window.start')

  df.printSchema()

  df.show(10, truncate=False)

except Exception as err:
  print(err)

root
 |-- window.start: timestamp (nullable = true)
 |-- requests: long (nullable = false)
 |-- avg: double (nullable = true)
 |-- max: double (nullable = true)
 |-- min: double (nullable = true)

+-------------------+--------+------------------+------+-----+
|window.start       |requests|avg               |max   |min  |
+-------------------+--------+------------------+------+-----+
|2016-12-06 08:58:30|483     |7.5934244306418215|46.849|0.013|
|2016-12-06 08:58:40|2611    |30.159845653006503|69.654|0.014|
|2016-12-06 08:58:50|5500    |38.52511163636371 |80.846|0.017|
|2016-12-06 08:59:00|6914    |38.53438212322824 |81.659|0.018|
|2016-12-06 08:59:10|6271    |32.96384978472328 |83.993|0.017|
|2016-12-06 08:59:20|5434    |17.29333143172616 |77.967|0.051|
|2016-12-06 08:59:30|8015    |11.210152214597631|67.441|0.056|
|2016-12-06 08:59:40|7947    |7.761815779539431 |65.706|0.914|
|2016-12-06 08:59:50|5983    |3.821664382416849 |54.29 |0.678|
|2016-12-06 09:00:00|6882    |8.649971519907023

2.3. Create an inverted index that, for each interval of 10 seconds, has a list of (unique) IPs executing accesses (to each URL).

In [74]:
#@title Exercise 2.3 - Inverted Index

from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('weblog2.2)').getOrCreate()

try :
  df_lines = spark.read.text('weblog.txt') \
                .withColumn('parts', split('value', ' '))

  # Extract array elements and cast to desired types
  df = df_lines.select(
    col('parts')[0].cast('timestamp').alias('timestamp'),
    col('parts')[1].alias('ip'),
    col('parts')[2].cast('integer').alias('status'),
    col('parts')[3].alias('method'),
    col('parts')[4].alias('url'),
    col('parts')[5].cast('double').alias('exec_time')
  )

  df = df.groupBy(window('timestamp', '10 seconds').start, 'url') \
          .agg(collect_set('ip').alias('ips')) \
          .orderBy('window.start')

  df.printSchema()

  df.show(10, truncate=False)

except Exception as err:
  print(err)

root
 |-- window.start: timestamp (nullable = true)
 |-- url: string (nullable = true)
 |-- ips: array (nullable = false)
 |    |-- element: string (containsNull = false)

+-------------------+----------------------+-------------------------------------+
|window.start       |url                   |ips                                  |
+-------------------+----------------------+-------------------------------------+
|2016-12-06 08:58:30|/codemove/01IX95N3AFP4|[120.52.73.98]                       |
|2016-12-06 08:58:30|/codemove/0GLNQSHCISWJ|[120.52.73.98]                       |
|2016-12-06 08:58:30|/codemove/1N80W0N2R36C|[120.52.73.97]                       |
|2016-12-06 08:58:30|/codemove/1U6HCG3V2S9D|[185.28.193.95]                      |
|2016-12-06 08:58:30|/codemove/2CEBGK8M78Y7|[192.241.151.220]                    |
|2016-12-06 08:58:30|/codemove/5Q9SRR2G46PJ|[120.52.73.98]                       |
|2016-12-06 08:58:30|/codemove/6GTXIA9YHX09|[120.52.73.98]                       